# 📈 Module 2: Linear Regression — Complete Guide
## Theory + Practical

---

## 📚 THEORY SECTION

### What is Linear Regression?

**Linear Regression** models the relationship between a dependent variable (y) and one or more independent variables (X) by fitting a **straight line** (or hyperplane in higher dimensions).

### Simple Linear Regression

```
y = w₀ + w₁·x + ε

Where:
  y    = target/output
  x    = input feature
  w₀   = intercept (bias term)
  w₁   = slope (weight/coefficient)
  ε    = error term (irreducible noise)
```

### Multiple Linear Regression

```
y = w₀ + w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ

In matrix form:  y = Xw
```

### How Does Linear Regression Learn? — Cost Function

The model learns by **minimizing the Mean Squared Error (MSE)**:

```
MSE = (1/n) Σ (yᵢ - ŷᵢ)²

Where:
  yᵢ  = actual value
  ŷᵢ  = predicted value
  n   = number of samples
```

### Optimization: Gradient Descent vs Closed-Form

| Method | Formula | When to Use |
|--------|---------|-------------|
| **Closed-Form (Normal Equation)** | `w = (XᵀX)⁻¹Xᵀy` | Small datasets, few features |
| **Gradient Descent** | `w = w - α·∇MSE` | Large datasets, many features |

### Gradient Descent — Step by Step

```
1. Initialize weights randomly (usually zeros)
2. Compute predictions: ŷ = Xw
3. Compute loss: MSE = (1/n)‖y - ŷ‖²
4. Compute gradient: ∇w = (-2/n) Xᵀ(y - ŷ)
5. Update weights:   w = w - α·∇w
6. Repeat until convergence
```

### Evaluation Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MSE** | `(1/n)Σ(y-ŷ)²` | Penalizes large errors heavily |
| **RMSE** | `√MSE` | Same unit as target, easy to interpret |
| **MAE** | `(1/n)Σ|y-ŷ|` | Robust to outliers |
| **R²** | `1 - SS_res/SS_tot` | % variance explained (0 to 1) |

### Assumptions of Linear Regression

1. **Linearity** — Relationship between X and y is linear
2. **Independence** — Observations are independent of each other
3. **Homoscedasticity** — Constant variance of errors
4. **Normality** — Residuals are normally distributed
5. **No Multicollinearity** — Features are not highly correlated

### Regularization

To prevent overfitting, we add penalty terms to the cost function:

| Method | Cost Function | Effect |
|--------|--------------|--------|
| **Ridge (L2)** | `MSE + λΣwᵢ²` | Shrinks all weights, keeps all features |
| **Lasso (L1)** | `MSE + λΣ|wᵢ|` | Sets some weights to zero (feature selection) |
| **ElasticNet** | `MSE + λ₁Σ|wᵢ| + λ₂Σwᵢ²` | Combines both |

---

## 🔬 PRACTICAL SECTION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.pipeline import make_pipeline

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('✅ Libraries loaded!')

### 🔬 Practical 1: Linear Regression From Scratch (Gradient Descent)

In [ ]:
# ============================================================
# PRACTICAL 1: Implement Linear Regression From Scratch
# ============================================================

class LinearRegressionFromScratch:
    """Linear Regression implemented using Gradient Descent."""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for i in range(self.n_iterations):
            # Forward pass (predictions)
            y_pred = X @ self.weights + self.bias
            
            # Compute MSE loss
            loss = np.mean((y - y_pred) ** 2)
            self.loss_history.append(loss)
            
            # Compute gradients
            dw = (-2/n_samples) * X.T @ (y - y_pred)
            db = (-2/n_samples) * np.sum(y - y_pred)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db
        
        return self
    
    def predict(self, X):
        return X @ self.weights + self.bias


# Generate data
X_raw = np.random.uniform(0, 10, (200, 1))
y_raw = 3.5 * X_raw.ravel() + 7 + np.random.randn(200) * 3

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Train from scratch
scratch_model = LinearRegressionFromScratch(learning_rate=0.01, n_iterations=500)
scratch_model.fit(X_tr, y_tr)

# Train sklearn model
sklearn_model = LinearRegression()
sklearn_model.fit(X_tr, y_tr)

print('From-Scratch Model:')
print(f'  Weight (w₁): {scratch_model.weights[0]:.4f}  (True: 3.5)')
print(f'  Bias   (w₀): {scratch_model.bias:.4f}  (True: 7.0)')
print(f'\nScikit-Learn Model:')
print(f'  Weight (w₁): {sklearn_model.coef_[0]:.4f}')
print(f'  Bias   (w₀): {sklearn_model.intercept_:.4f}')

# Plot training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(scratch_model.loss_history, color='#E74C3C', linewidth=2)
axes[0].set_title('Training Loss (MSE) over Iterations', fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE Loss')
axes[0].grid(True, alpha=0.3)

X_plot_line = np.linspace(0, 10, 100).reshape(-1, 1)
axes[1].scatter(X_te, y_te, color='#3498DB', alpha=0.6, s=40, label='Test Data')
axes[1].plot(X_plot_line, scratch_model.predict(X_plot_line), 'r-', 
             linewidth=2.5, label='Our Model (GD)')
axes[1].plot(X_plot_line, sklearn_model.predict(X_plot_line), 'g--', 
             linewidth=2, label='Sklearn Model')
axes[1].set_title('Model Fit Comparison', fontweight='bold')
axes[1].set_xlabel('X')
axes[1].set_ylabel('y')
axes[1].legend()

plt.tight_layout()
plt.show()

### 🔬 Practical 2: California Housing — Real Dataset

In [ ]:
# ============================================================
# PRACTICAL 2: Real Dataset — California Housing Prices
# ============================================================

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target  # Median house value in $100k

print(f'Dataset shape: {df.shape}')
print(f'\nFeature descriptions:')
desc = {
    'MedInc': 'Median income in block group',
    'HouseAge': 'Median house age',
    'AveRooms': 'Average rooms per household',
    'AveBedrms': 'Average bedrooms per household',
    'Population': 'Block group population',
    'AveOccup': 'Average household size',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude'
}
for feat, description in desc.items():
    print(f'  {feat:12s}: {description}')

print(f'\nTarget: Median house value (in $100,000)')
df.describe().round(2)

In [ ]:
# EDA — Correlation with target
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation bar chart
correlations = df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values()
colors = ['#E74C3C' if c < 0 else '#2ECC71' for c in correlations]
axes[0].barh(correlations.index, correlations.values, color=colors, edgecolor='black', alpha=0.8)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Feature Correlation with House Price', fontweight='bold')
axes[0].set_xlabel('Pearson Correlation Coefficient')
for i, (val, feat) in enumerate(zip(correlations.values, correlations.index)):
    axes[0].text(val + 0.01 if val > 0 else val - 0.01, i, f'{val:.2f}', 
                 va='center', fontsize=9)

# MedInc vs Price scatter
axes[1].scatter(df['MedInc'], df['MedHouseVal'], alpha=0.2, s=10, color='steelblue')
m, b = np.polyfit(df['MedInc'], df['MedHouseVal'], 1)
x_line = np.linspace(df['MedInc'].min(), df['MedInc'].max(), 100)
axes[1].plot(x_line, m*x_line + b, 'r-', linewidth=2, label='Best Fit Line')
axes[1].set_title('Income vs House Price', fontweight='bold')
axes[1].set_xlabel('Median Income')
axes[1].set_ylabel('Median House Value ($100k)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Build and evaluate Linear Regression model
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Compare models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (L2, α=1.0)': Ridge(alpha=1.0),
    'Lasso (L1, α=0.01)': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5)
}

print(f'{"Model":<25} {"Train R²":<12} {"Test R²":<12} {"RMSE":<10}')
print('-' * 62)

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred_tr = model.predict(X_train_s)
    y_pred_te = model.predict(X_test_s)
    
    train_r2 = r2_score(y_train, y_pred_tr)
    test_r2  = r2_score(y_test, y_pred_te)
    rmse     = np.sqrt(mean_squared_error(y_test, y_pred_te))
    
    results[name] = {'train_r2': train_r2, 'test_r2': test_r2, 'rmse': rmse, 
                     'model': model, 'y_pred': y_pred_te}
    print(f'{name:<25} {train_r2:<12.4f} {test_r2:<12.4f} {rmse:<10.4f}')

In [ ]:
# Residual analysis — KEY diagnostic for regression
best_model = results['Linear Regression']
residuals = y_test.values - best_model['y_pred']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residual Analysis — Linear Regression Diagnostics', fontsize=14, fontweight='bold')

# Predicted vs Actual
axes[0, 0].scatter(best_model['y_pred'], y_test, alpha=0.3, s=15, color='steelblue')
pmin, pmax = best_model['y_pred'].min(), best_model['y_pred'].max()
axes[0, 0].plot([pmin, pmax], [pmin, pmax], 'r--', linewidth=2)
axes[0, 0].set_title('Predicted vs Actual')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')

# Residuals vs Predicted (Homoscedasticity check)
axes[0, 1].scatter(best_model['y_pred'], residuals, alpha=0.3, s=15, color='#E74C3C')
axes[0, 1].axhline(0, color='black', linewidth=2)
axes[0, 1].set_title('Residuals vs Predicted\n(Should be random scatter)')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Residuals')

# Residual Distribution (Normality check)
axes[1, 0].hist(residuals, bins=50, color='#2ECC71', edgecolor='black', alpha=0.8)
axes[1, 0].set_title('Residual Distribution\n(Should be bell-shaped)')
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_ylabel('Frequency')

# Feature importance (coefficients)
lr_model = results['Linear Regression']['model']
coef_df = pd.DataFrame({'Feature': housing.feature_names, 'Coefficient': lr_model.coef_})
coef_df = coef_df.sort_values('Coefficient')
colors_coef = ['#E74C3C' if c < 0 else '#2ECC71' for c in coef_df['Coefficient']]
axes[1, 1].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef, edgecolor='black')
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('Feature Coefficients (Scaled)')
axes[1, 1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

### 🔬 Practical 3: Regularization — Ridge vs Lasso

In [ ]:
# ============================================================
# PRACTICAL 3: Regularization Paths
# ============================================================

alphas = np.logspace(-3, 3, 50)

ridge_coefs, lasso_coefs = [], []
ridge_r2, lasso_r2 = [], []

for alpha in alphas:
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_s, y_train)
    ridge_coefs.append(ridge.coef_)
    ridge_r2.append(r2_score(y_test, ridge.predict(X_test_s)))
    
    # Lasso
    lasso = Lasso(alpha=alpha, max_iter=5000)
    lasso.fit(X_train_s, y_train)
    lasso_coefs.append(lasso.coef_)
    lasso_r2.append(r2_score(y_test, lasso.predict(X_test_s)))

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Regularization: Ridge vs Lasso', fontsize=14, fontweight='bold')

# Ridge coefficient paths
for i, feat in enumerate(housing.feature_names):
    axes[0, 0].semilogx(alphas, ridge_coefs[:, i], label=feat, linewidth=2)
axes[0, 0].set_title('Ridge (L2): Coefficient Paths')
axes[0, 0].set_xlabel('Alpha (log scale)')
axes[0, 0].set_ylabel('Coefficient Value')
axes[0, 0].legend(fontsize=7, loc='upper right')
axes[0, 0].axhline(0, color='black', linewidth=0.8)

# Lasso coefficient paths
for i, feat in enumerate(housing.feature_names):
    axes[0, 1].semilogx(alphas, lasso_coefs[:, i], label=feat, linewidth=2)
axes[0, 1].set_title('Lasso (L1): Coefficient Paths\n(Notice: coefficients go to ZERO!)')
axes[0, 1].set_xlabel('Alpha (log scale)')
axes[0, 1].set_ylabel('Coefficient Value')
axes[0, 1].legend(fontsize=7, loc='upper right')
axes[0, 1].axhline(0, color='black', linewidth=0.8)

# R2 vs Alpha
axes[1, 0].semilogx(alphas, ridge_r2, 'b-', linewidth=2, label='Ridge')
axes[1, 0].set_title('Ridge: Test R² vs Alpha')
axes[1, 0].set_xlabel('Alpha')
axes[1, 0].set_ylabel('Test R²')
axes[1, 0].axvline(alphas[np.argmax(ridge_r2)], color='red', linestyle='--', 
                    label=f'Best α={alphas[np.argmax(ridge_r2)]:.3f}')
axes[1, 0].legend()

axes[1, 1].semilogx(alphas, lasso_r2, 'r-', linewidth=2, label='Lasso')
axes[1, 1].set_title('Lasso: Test R² vs Alpha')
axes[1, 1].set_xlabel('Alpha')
axes[1, 1].set_ylabel('Test R²')
axes[1, 1].axvline(alphas[np.argmax(lasso_r2)], color='blue', linestyle='--',
                    label=f'Best α={alphas[np.argmax(lasso_r2)]:.3f}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print('\n📌 Key Insight:')
print('  Ridge: All coefficients shrink toward 0 but never reach exactly 0')
print('  Lasso: Coefficients can become exactly 0 (automatic feature selection!)')

## 🎓 Summary

| Concept | Key Takeaway |
|---------|-------------|
| Linear Regression | Fits line by minimizing MSE |
| Normal Equation | Exact solution: `w = (XᵀX)⁻¹Xᵀy` |
| Gradient Descent | Iterative optimization for large data |
| R² Score | How much variance is explained (1.0 = perfect) |
| Ridge (L2) | Shrinks weights, keeps all features |
| Lasso (L1) | Sets some weights to exactly zero (feature selection) |
| Data Leakage | NEVER fit scaler/model on test data |

## 📖 Next: Logistic Regression & Classification